In [4]:
!pip install coremltools onnx addict jiwer datasets
import torch
from transformers import AutoModel, AutoProcessor, AutoTokenizer
import datasets
import onnx
import coremltools as ct
from pathlib import Path
from abc import ABC, abstractmethod
from PIL import Image
import numpy as np
from jiwer import cer, wer
from time import time
from tqdm import tqdm
import pandas as pd

import warnings
warnings.filterwarnings(action='ignore')

In [15]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# 1. Classes for each model

In [16]:
class AbstractModel(ABC):
    def __init__(self, model_id, output_dir):
        self.model_id = model_id
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True, parents=True)
        self.model = None
        self.processor = None

    @property
    @abstractmethod
    def model_name(self):
        """Getter for model name"""
        pass

    @abstractmethod
    def load_model(self):
        """Method to import model & set it on eval mode"""
        pass

    @abstractmethod
    def prepare_for_inference(self):
        """Loading model, setting to active device"""
        pass

    @abstractmethod
    def predict(self, image_path, prompt=None):
        """
        Run inference on single image
        Returns: (text, inference_time)
        """
        pass

    def export_to_onnx(self):
        """Setting up model to onnx format - может не работать для VLM"""
        raise NotImplementedError(
            f"ONNX export for {self.model_name} requires custom implementation"
        )

    def to_coreml(self):
        """Exporting model to .mlmodel fmt - может не работать для VLM"""
        raise NotImplementedError(
            f"CoreML export for {self.model_name} requires custom implementation"
        )

### PaddleOCR

In [17]:
# 3 448 448
class PaddleModel(AbstractModel):
    @property
    def model_name(self):
        return "PaddleOCR-VL"

    def load_model(self):
        self.processor = AutoProcessor.from_pretrained(
            self.model_id,
            trust_remote_code=True
        )

        self.model = AutoModel.from_pretrained(
            self.model_id,
            trust_remote_code=True,
            torch_dtype=torch.bfloat16,
            device_map="auto"
        )

        self.model.eval()
        print(f"Loaded {self.model_id}")

    def prepare_for_inference(self):
        if self.model is None:
            self.load_model()

        self.model = self.model.to(DEVICE)
        print(f"Model ready for inference {self.model_id}")

    def predict(self, image_path, prompt="Extract all text from this image"):
        if self.model is None:
            self.prepare_for_inference()

        start = time()
        image = Image.open(image_path).convert('RGB')

        # PaddleOCR-VL специфичный формат
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": prompt}
                ]
            }
        ]

        inputs = self.processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=False
            )

        text = self.processor.decode(outputs[0], skip_special_tokens=True)
        inference_time = time() - start

        return text, inference_time

### QwenV3

In [18]:
from transformers import Qwen2VLForConditionalGeneration, Qwen2VLProcessor

# 3 448 448
class Qwen3VLModel(AbstractModel):
    @property
    def model_name(self):
        return "Qwen3-VL-2B"


    def load_model(self):
        self.processor = Qwen2VLProcessor.from_pretrained(
            self.model_id,
            trust_remote_code=True
        )
        self.model = Qwen2VLForConditionalGeneration.from_pretrained(
            self.model_id,
            trust_remote_code=True,
            torch_dtype=torch.bfloat16,
            device_map="auto"
        )

        self.model.eval()
        print(f"Model loaded {self.model_id}")


    def prepare_for_inference(self):
        if self.model is None:
            self.load_model()

        self.model = self.model.to("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Model ready for inference {self.model_id}")


    def predict(self, image_path, prompt="Extract all text"):
        if self.model is None:
            self.prepare_for_inference()

        start = time()
        image = Image.open(image_path).convert('RGB')

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": prompt}
                ]
            }
        ]

        text_prompt = self.processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = self.processor(
            text=[text_prompt],
            images=[image],
            padding=True,
            return_tensors="pt"
        ).to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=False
            )

        text = self.processor.decode(outputs[0], skip_special_tokens=True)
        inference_time = time() - start

        return text, inference_time

### LightOn

In [19]:
class LightOnModel(AbstractModel):
    @property
    def model_name(self):
        return "LightOn-OCR"

    def load_model(self):
        self.processor = AutoProcessor.from_pretrained(
            self.model_id,
            trust_remote_code=True
        )
        self.model = AutoModel.from_pretrained(
            self.model_id,
            trust_remote_code=True,
            torch_dtype=torch.bfloat16,
            device_map="auto"
        )
        self.model.eval()
        print(f"Loaded {self.model_id}")

    def prepare_for_inference(self):
        if self.model is None:
            self.load_model()
        self.model = self.model.to("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Model ready for inference {self.model_id}")

    def predict(self, image_path, prompt=None):
        if self.model is None:
            self.prepare_for_inference()

        start = time()

        image = Image.open(image_path).convert('RGB')
        inputs = self.processor(images=image, return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_new_tokens=512)

        text = self.processor.decode(outputs[0], skip_special_tokens=True)
        inference_time = time() - start

        return text, inference_time

### DeepSeek AI

In [20]:
class DeepseekModel(AbstractModel):
    @property
    def model_name(self):
        return "DeepSeek-OCR"

    def load_model(self):
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_id,
            trust_remote_code=True
        )
        self.model = AutoModel.from_pretrained(
            self.model_id,
            trust_remote_code=True,
            torch_dtype=torch.bfloat16,
            device_map="auto"
        )
        self.model.eval()
        print(f"Loaded {self.model_id}")

    def prepare_for_inference(self):
        if self.model is None:
            self.load_model()
        self.model = self.model.to("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Model ready for inference {self.model_id}")

    def predict(self, image_path, prompt=None):
        """ai written"""
        if self.model is None:
            self.prepare_for_inference()

        start = time()

        image = Image.open(image_path).convert('RGB')
        # DeepSeek-специфичный API
        inputs = self.model.prepare_inputs(image, self.tokenizer)
        inputs = {k: v.to(self.model.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_new_tokens=512)

        text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        inference_time = time() - start

        return text, inference_time

## Actual fabric wtf

In [27]:
class ModelFactory:
    """Fabric interface for controlling all models"""

    AVAILABLE_MODELS = {}

    @classmethod
    def register_model(cls, name, model_class, model_id):
        """Add model to available in this fabric"""
        cls.AVAILABLE_MODELS[name] = {
            "class": model_class,
            "model_id": model_id,
        }


    @classmethod
    def list_models(cls):
        """List available models"""
        for name, info in cls.AVAILABLE_MODELS.items():
            print(name)
            print(info['model_id'], end='\n\n')

    @classmethod
    def create_model(cls, model_name, output_dir = "./models"):
        """Creating an instance of model"""

        if model_name not in cls.AVAILABLE_MODELS:
            raise ValueError(
                f"Unknown model: {model_name}. "
            )

        model_info = cls.AVAILABLE_MODELS[model_name]
        model_class = model_info["class"]

        model = model_class(
            model_id=model_info["model_id"],
            output_dir=output_dir
        )

        print(f"{model_name} created successfully")
        return model

## Registrating all available models

In [28]:
def register_all_models():
    """Registrating all available models"""

    ModelFactory.register_model(
        name="paddle",
        model_class=PaddleModel,
        model_id="PaddlePaddle/PaddleOCR-VL",
    )

    ModelFactory.register_model(
        name="qwen3-2b",
        model_class=Qwen3VLModel,
        model_id="Qwen/Qwen3-VL-2B-Instruct",
    )

    ModelFactory.register_model(
        name="lighton",
        model_class=LightOnModel,
        model_id="lightonai/LightOnOCR-0.9B-32k-1025",
    )

    ModelFactory.register_model(
        name="deepseek",
        model_class=DeepseekModel,
        model_id="deepseek-ai/DeepSeek-OCR",
    )

validating... (ai powered bs)

In [29]:
register_all_models()
ModelFactory.list_models()
model = ModelFactory.create_model("qwen3-2b")

paddle
PaddlePaddle/PaddleOCR-VL

qwen3-2b
Qwen/Qwen3-VL-2B-Instruct

lighton
lightonai/LightOnOCR-0.9B-32k-1025

deepseek
deepseek-ai/DeepSeek-OCR

qwen3-2b created successfully


In [11]:
results = validate_models(
    model_factory=ModelFactory,
    save_results=True,
    output_file="model_validation_results.csv"
)


🚀 MODEL VALIDATION SUITE

Found 4 models to validate:
  • paddle
  • qwen3-2b
  • lighton
  • deepseek


🔍 Validating: paddle
  [1/5] Creating model instance...

  ❌ paddle validation FAILED!
  Error: 'PaddleModel'

  Traceback:

🔍 Validating: qwen3-2b
  [1/5] Creating model instance...

  ❌ qwen3-2b validation FAILED!
  Error: 'Qwen3VLModel'

  Traceback:

🔍 Validating: lighton
  [1/5] Creating model instance...

  ❌ lighton validation FAILED!
  Error: 'LightOnModel'

  Traceback:

🔍 Validating: deepseek
  [1/5] Creating model instance...

  ❌ deepseek validation FAILED!
  Error: 'DeepseekModel'

  Traceback:

📊 VALIDATION SUMMARY

Total models: 4
✅ Passed: 0
❌ Failed: 4
Success rate: 0.0%

----------------------------------------------------------------------
Detailed Results:
----------------------------------------------------------------------
model_name   status creation_time load_time inference_time device           error
    paddle ❌ FAILED          None      None           No

Traceback (most recent call last):
  File "/tmp/ipython-input-159119974.py", line 87, in validate_model
    model = model_factory.create_model(model_name, self.output_dir)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipython-input-1669485455.py", line 37, in create_model
    model_class = globals()[model_info['class']]
                  ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^
KeyError: 'PaddleModel'
Traceback (most recent call last):
  File "/tmp/ipython-input-159119974.py", line 87, in validate_model
    model = model_factory.create_model(model_name, self.output_dir)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipython-input-1669485455.py", line 37, in create_model
    model_class = globals()[model_info['class']]
                  ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^
KeyError: 'Qwen3VLModel'
Traceback (most recent call last):
  File "/tmp/ipython-input-159119974.py", line 87, in validate_model
    model = model_factory.create_mo

# 2. Benchmark class

In [6]:
class BenchmarkLoader:
    @staticmethod
    def load_mws_vision(task_type=None, split='test'):
        """MWS Vision Bench"""
        dataset = datasets.load_dataset("MTSAIR/MWS-Vision-Bench", split=split)

        if task_type:
            dataset = dataset.filter(lambda x: x["task_type"] == task_type)

        samples = []
        for item in dataset:
            samples.append({
                "image": item["image"],
                "ground_truth": item["ground_truth"],
                "task_type": item.get("task_type"),
                "question": item.get("question", None),
                "metadata": item.get("metadata", {})
            })

        return samples

    @staticmethod
    def load_labtabvqa(split="test"):
        """MERA LabTab VQA"""
        dataset = datasets.load_dataset("MERA-evaluation/LabTabVQA", split=split)

        samples = []
        for item in dataset:
            options = {
                'A': item['inputs']['option_a'],
                'B': item['inputs']['option_b'],
                'C': item['inputs']['option_c'],
                'D': item['inputs']['option_d'],
                'E': item['inputs']['option_e'],
                'F': item['inputs']['option_f'],
                'G': item['inputs']['option_g'],
            }

            samples.append({
                "image": item['inputs']['image'],
                "question": item['inputs']['question'],
                "options": options,
                "correct_answer": item['outputs'],  # буква A-G
                "instruction": item.get('instruction', ''),
                "meta": item.get('meta', {})
            })

        return samples